[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/04_attention_mechanism/04_attention_mechanism.ipynb)

# 04. Attention Mechanism Deep Dive

**This notebook covers:**
- Scaled dot-product self-attention from first principles
- Multi-head attention implementation
- Attention heatmap visualization
- Cross-attention for multimodal fusion
- Numerical step-by-step trace with real numbers

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/01_Multimodal_Foundations/04_attention_mechanism"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Scaled Dot-Product Attention

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Division by $\sqrt{d_k}$ keeps dot-product variance at 1 as $d_k$ grows.


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# Numerical trace: 3 tokens, d_k=4 (from README worked example)
Q = torch.tensor([[[1., 0., 1., 0.], [0., 1., 0., 1.], [1., 1., 0., 0.]]])
K = torch.tensor([[[1., 1., 0., 0.], [0., 0., 1., 1.], [1., 0., 1., 0.]]])
V = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]])

raw = Q @ K.transpose(-2, -1)
scaled = raw / (4 ** 0.5)
print("QK^T / sqrt(d_k) =\n", scaled[0].numpy().round(3))
out, attn = scaled_dot_product_attention(Q, K, V)
print("\nAttention weights (row 0):", attn[0, 0].numpy().round(3), " sum=", attn[0, 0].sum().item())
print("Output token 0:", out[0, 0].numpy().round(3))

## 2. Multi-Head Attention

Each head learns a different relational pattern; outputs are concatenated and projected.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn_output), attn_weights

mha = MultiHeadAttention(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out, w = mha(x, x, x)
print(f"Input {x.shape} -> Output {out.shape}, attn {w.shape}")
count_parameters(mha)

## 3. Attention Heatmap Visualization


In [ ]:
tokens = ['CLS', 'a', 'cat', 'sits']
x = torch.randn(1, len(tokens), 64)
with torch.no_grad():
    _, attn = mha(x, x, x)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for h, ax in enumerate(axes):
    im = ax.imshow(attn[0, h].numpy(), cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)), labels=tokens)
    ax.set_yticks(range(len(tokens)), labels=tokens)
    ax.set_title(f'Head {h+1}')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Multi-Head Self-Attention Weights')
plt.tight_layout()
plt.show()

## 4. Cross-Attention (Multimodal Fusion)

Text tokens **query** image patch keys/values: $\text{CrossAttn}(Z_{txt}, Z_{img})$.


In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, d_model, num_heads=4):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)

    def forward(self, query_seq, kv_seq):
        return self.mha(query_seq, kv_seq, kv_seq)

cross = CrossAttention(64, num_heads=4)
text = torch.randn(1, 4, 64)   # 4 text tokens
image = torch.randn(1, 16, 64)  # 16 patch tokens
fused, cross_attn = cross(text, image)
print(f"Text {text.shape} attends to image {image.shape} -> {fused.shape}")
print(f"Cross-attn matrix shape: {cross_attn.shape}  (heads, text, patches)")

## 5. Causal Masking Preview

Decoder self-attention masks future positions with $M_{ij} = -\infty$ when $i < j$.


In [ ]:
seq_len = 5
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
x = torch.randn(1, seq_len, 64)
with torch.no_grad():
    _, masked_attn = mha(x, x, x, mask=causal_mask.unsqueeze(0).unsqueeze(0))
print("Causal attention (upper triangle should be ~0):")
print(masked_attn[0, 0].numpy().round(3))

## 6. Complexity Analysis

Self-attention over $n$ tokens costs $O(n^2 d)$ per layer. Cross-attention between $T$ text and $N$ image tokens costs $O(T \cdot N \cdot d)$.


In [ ]:
def attention_flops(n_tokens, d_model, n_heads):
    d_k = d_model // n_heads
    qkv = 3 * n_tokens * d_model * d_model
    scores = n_tokens * n_tokens * d_k * n_heads
    weighted = n_tokens * n_tokens * d_k * n_heads
    out_proj = n_tokens * d_model * d_model
    return qkv + scores + weighted + out_proj

for n in [64, 128, 256]:
    flops = attention_flops(n, 256, 4)
    print(f"n={n:3d} tokens -> ~{flops/1e6:.2f}M multiply-adds per layer")

## Summary

- Built scaled dot-product and multi-head attention from scratch
- Visualized attention heatmaps and cross-modal fusion
- Traced a numerical example matching the README derivation

**Next:** [05_transformer_architecture](../05_transformer_architecture/05_transformer_architecture.ipynb)
